In [1]:
# !python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

In [2]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
# if you got an API key
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")

# Now for each of the ten projects you were assigned get commits, e.g.
projects = ['brainglobe_brainrender',
    'openworm_sibernetic',
    'gnuradio_gr-inspector',
    'adibender_pammtools',
    'spacetelescope_drizzlepac',
    'unity-technologies_synthdet',
    'cp2k_cp2k',
    'rub-nds_tls-attacker',
    'gigascience_gigadb-website',
    'atomistic-machine-learning_schnetpack']
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,00234f276e5bf75273d66e54ff86ff2b4d424d6e,brainglobe_brainrender


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

In [4]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

NameError: name 'commit_data' is not defined

In [ ]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame({
    'project_wocid': df_commit_data['project'],
    'commit_sha1':   df_commit_data['commit'],
    'author':        df_commit_data['author'],
    'time':          df_commit_data['author_time'],
    'commit message': df_commit_data['message'],
})

In [ ]:
# check if it has the right content
dfinf.head(1)

In [ ]:
#mode a means append, so you have all your projects in the same file
yournetid='pstorch1'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', header=False)

### brainglobe_brainrender
- GitHub: 673 stars, 108 forks, last commit 2026/09/28
- WoC: 1519 commits, 49 authors, max time 1776246364 (2026-04-15), min time 1568885229 (2019-09-19)

### openworm_sibernetic
- GitHub: 392 stars, 115 forks, last commit 2026/05/28
- WoC: 1048 commits, 56 authors, max time 1760104573 (2025-10-10), min time 1338455836 (2012-05-31)

### gnuradio_gr-inspector
- GitHub: 319 stars, 104 forks, last commit 2025/03/03
- WoC: 539 commits, 20 authors, max time 1741029916 (2025-03-03), min time 1461363769 (2016-04-22)

### adibender_pammtools
- GitHub: 51 stars, 13 forks, last commit 2026/08/13
- WoC: 1445 commits, 25 authors, max time 1776705654 (2026-04-20), min time 1483636805 (2017-01-05)

### spacetelescope_drizzlepac
- GitHub: 58 stars, 42 forks, last commit 2026/09/14
- WoC: 9282 commits, 68 authors, max time 1762374727 (2025-11-05), min time 1229621642 (2008-12-18)

### unity-technologies_synthdet
- GitHub: 402 stars, 58 forks, last commit 2024/12/05
- WoC: 421 commits, 32 authors, max time 1733458700 (2024-12-06), min time 1585249473 (2020-03-26)

### cp2k_cp2k
- GitHub: 1212 stars, 504 forks, last commit 2026/09/29
- WoC: 50549 commits, 330 authors, max time 1762600587 (2025-11-08), min time 993571518 (2001-06-26)

### rub-nds_tls-attacker
- GitHub: 881 stars, 146 forks, last commit 2026/07/24
- WoC: 9115 commits, 189 authors, max time 1759424521 (2025-10-02), min time 1427385713 (2015-03-26)

### gigascience_gigadb-website
- GitHub: 14 stars, 15 forks, last commit 2026/06/09
- WoC: 14468 commits, 40 authors, max time 1760640135 (2025-10-16), min time 1459997834 (2016-04-07)

### atomistic-machine-learning_schnetpack
- GitHub: 941 stars, 251 forks, last commit 2026/09/29
- WoC: 3379 commits, 83 authors, max time 1759737854 (2025-10-06), min time 1535989475 (2018-09-03)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github